In [1]:
import pandas as pd
import folium
from folium.plugins import MarkerCluster
from collections import defaultdict


df = pd.read_csv("individual model_data 2016_2020 greater manchester.csv")

# Create a map (with the center point located near Manchester)
m = folium.Map(location=[53.4808, -2.2426], zoom_start=11)

# Add points to the map (with the option to use MarkerCluster for aggregation)
marker_cluster = MarkerCluster().add_to(m)

for idx, row in df.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=1,
        color='black',
        fill=True,
        fill_opacity=0.7
    ).add_to(marker_cluster)



# Save the map to an HTML file
m.save("map_with_paths.html")


In [2]:
import pandas as pd
import folium

# Read data
file_path = "individual model_data 2016_2020 greater manchester.csv"  
df = pd.read_csv(file_path)

# Remove missing coordinates
df = df.dropna(subset=['latitude', 'longitude'])

# Create Map
m = folium.Map(location=[53.4808, -2.2426], zoom_start=12, tiles="OpenStreetMap")

# Add larger black dots (without aggregation)
for _, row in df.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=8,            
        color='black',
        fill=True,
        fill_color='black',
        fill_opacity=0.75,
        weight=0
    ).add_to(m)

# save as HTML
m.save("black_dots_larger.html")


In [3]:
import pandas as pd
import folium
from folium.plugins import HeatMap
from folium import FeatureGroup, LayerControl

# Read data
df = pd.read_csv("individual model_data 2016_2020 greater manchester.csv")

# Only retain the data from 2016 to 2020
df_filtered = df[df['year'].between(2016, 2020)]

# Remove records with missing latitude and longitude coordinates
df_filtered = df_filtered.dropna(subset=['latitude', 'longitude'])

# Initialize the map center point (the center of Manchester)
map_center = [53.4808, -2.2426]
m = folium.Map(location=map_center, zoom_start=11)

# Obtain the unique values for the year and the severity of the accident
years = sorted(df_filtered['year'].unique())
severity_levels = df_filtered['slightsevere'].unique()

# Create heat map layers for each year and severity level
for year in years:
    for severity in severity_levels:
        yearly_data = df_filtered[(df_filtered['year'] == year) & (df_filtered['slightsevere'] == severity)]
        heat_points = yearly_data[['latitude', 'longitude']].values.tolist()
        if heat_points:
            layer_name = f"{year} - {severity}"
            fg = FeatureGroup(name=layer_name)
            HeatMap(heat_points, radius=12, blur=15).add_to(fg)
            fg.add_to(m)

# Add Layer Controller
LayerControl(collapsed=False).add_to(m)

# Save as HTML file
m.save("manchester_heatmap_2016_2020.html")
